# Terra · Clima agrícola de Morelos (NASA Giovanni)

Series mensuales **2000–2026** promediadas sobre Morelos (bbox `-99.5, 18.33, -98.63, 19.14`) a partir de modelos de superficie terrestre de NASA:

| Fuente | Resolución | Para qué sirve en agricultura |
|---|---|---|
| **FLDAS Noah** (FEWS NET Land Data Assimilation System) | 0.1°, mensual | Lluvia, temperatura, evapotranspiración, humedad y temperatura del suelo, radiación, escorrentía. Diseñado para alerta temprana de seguridad alimentaria |
| **FLDAS anomalías** | 0.1°, mensual | Desviación respecto a la climatología: detectar sequías y olas de calor |
| **GLDAS Noah 2.1** | 0.25°, mensual | Evaporación potencial (demanda de riego) y humedad en la zona radicular |

**Cómo cargar los datos:** sube `morelos_clima_mensual_2000_2026.csv` (carpeta `Terra/datos_procesados`) con la celda siguiente, o ponlo en Google Drive en `MyDrive/Terra/datos_procesados/`.

In [ ]:
import os, pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid")

ARCHIVO = "morelos_clima_mensual_2000_2026.csv"
RUTA_DRIVE = "/content/drive/MyDrive/Terra/datos_procesados/" + ARCHIVO

try:
    from google.colab import drive, files
    try:
        drive.mount("/content/drive")
    except Exception as e:
        print("Drive no montado:", e)
    candidatas = [RUTA_DRIVE, "/content/drive/MyDrive/Terra/" + ARCHIVO]
    ruta = next((p for p in candidatas if os.path.exists(p)), None)
    if ruta is None:
        print("No está en Drive; selecciona el CSV para subirlo:")
        ruta = list(files.upload().keys())[0]
except ImportError:          # fuera de Colab
    ruta = os.path.join("..", "datos_procesados", ARCHIVO)

df = pd.read_csv(ruta, parse_dates=["fecha"]).set_index("fecha").asfreq("MS")
print(df.shape); df.tail()

## 1. Calidad de datos

In [ ]:
display(df.describe().T.round(3))
print("Faltantes por variable:"); print(df.isna().sum())

## 2. Series de tiempo principales

In [ ]:
vars_ = ["lluvia_mm_mes","temp_aire_C","evapotransp_mm_mes","hum_suelo_10_40cm_m3m3","radiacion_solar_Wm2","evap_potencial_mm_dia"]
vars_ = [v for v in vars_ if v in df]
fig, axes = plt.subplots(len(vars_), 1, figsize=(14, 2.6*len(vars_)), sharex=True)
for ax, v in zip(axes, vars_):
    ax.plot(df.index, df[v], lw=1)
    ax.plot(df.index, df[v].rolling(12, center=True).mean(), lw=2, label="media móvil 12 m")
    ax.set_ylabel(v, fontsize=8)
axes[0].legend(); plt.tight_layout(); plt.show()

## 3. Climatología mensual (calendario agrícola)
La temporada de lluvias (jun–oct) define la siembra de temporal de maíz, frijol y sorgo; la caña y hortalizas dependen de riego en secas.

In [ ]:
clim = df.groupby(df.index.month).mean()
clim.index = ["Ene","Feb","Mar","Abr","May","Jun","Jul","Ago","Sep","Oct","Nov","Dic"]
fig, ax1 = plt.subplots(figsize=(12,5))
ax1.bar(clim.index, clim["lluvia_mm_mes"], color="steelblue", label="Lluvia (mm/mes)")
if "evapotransp_mm_mes" in clim:
    ax1.plot(clim.index, clim["evapotransp_mm_mes"], "g-o", label="Evapotranspiración (mm/mes)")
if "evap_potencial_mm_dia" in clim:
    ax1.plot(clim.index, clim["evap_potencial_mm_dia"]*30.4, "r--o", label="Evap. potencial (mm/mes)")
ax1.set_ylabel("mm/mes"); ax1.legend(loc="upper left")
ax2 = ax1.twinx(); ax2.plot(clim.index, clim["temp_aire_C"], "k:s", label="Temp. aire (°C)"); ax2.set_ylabel("°C"); ax2.grid(False)
plt.title("Climatología mensual Morelos 2000–2026"); plt.show()

## 4. Balance hídrico y estrés hídrico
Déficit = lluvia − evaporación potencial. Valores negativos sostenidos indican necesidad de riego.

In [ ]:
if "evap_potencial_mm_dia" in df:
    df["deficit_hidrico_mm"] = df["lluvia_mm_mes"] - df["evap_potencial_mm_dia"] * df.index.days_in_month
    anual = df["deficit_hidrico_mm"].resample("YS").sum()
    anual.index = anual.index.year
    anual.plot(kind="bar", figsize=(14,4), color=np.where(anual<anual.median(),"indianred","seagreen"))
    plt.title("Balance hídrico anual (lluvia − ETP), mm"); plt.ylabel("mm"); plt.show()

## 5. Anomalías: sequías y años calurosos

In [ ]:
anom = [c for c in df if c.startswith("anom_")]
if anom:
    fig, axes = plt.subplots(len(anom), 1, figsize=(14, 2.8*len(anom)), sharex=True)
    for ax, v in zip(np.atleast_1d(axes), anom):
        s = df[v]
        ax.bar(s.index, s, width=25, color=np.where(s<0, "indianred", "steelblue"))
        ax.set_ylabel(v, fontsize=8)
    plt.tight_layout(); plt.show()

# Índice estandarizado de lluvia (SPI simplificado, 3 meses)
lluvia3 = df["lluvia_mm_mes"].rolling(3).sum()
spi3 = lluvia3.groupby(lluvia3.index.month).transform(lambda x: (x - x.mean())/x.std())
plt.figure(figsize=(14,3)); plt.fill_between(spi3.index, spi3, 0, where=spi3<0, color="indianred"); plt.fill_between(spi3.index, spi3, 0, where=spi3>=0, color="steelblue")
plt.axhline(-1, ls="--", c="k", lw=.8); plt.title("SPI-3 aproximado (< −1 = sequía moderada)"); plt.show()
df["spi3"] = spi3

## 6. Correlaciones entre variables

In [ ]:
plt.figure(figsize=(11,9))
sns.heatmap(df.corr(numeric_only=True), cmap="RdBu_r", center=0, annot=True, fmt=".2f", annot_kws={"size":7})
plt.title("Correlación de Pearson"); plt.show()

## 7. Tendencias de largo plazo (Mann-Kendall)

In [ ]:
!pip -q install pymannkendall
import pymannkendall as mk
filas = []
for v in ["lluvia_mm_mes","temp_aire_C","evapotransp_mm_mes","hum_suelo_10_40cm_m3m3","temp_suelo_0_10cm_C"]:
    if v in df:
        r = mk.seasonal_test(df[v].dropna(), period=12)
        filas.append([v, r.trend, round(r.p,4), round(r.slope,4)])
pd.DataFrame(filas, columns=["variable","tendencia","p-valor","pendiente por año"])

## 8. Descomposición estacional

In [ ]:
from statsmodels.tsa.seasonal import STL
res = STL(df["temp_aire_C"].interpolate(), period=12).fit()
res.plot(); plt.show()

## 9. Pronóstico de lluvia y temperatura (SARIMA, 12 meses)

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
for v in ["lluvia_mm_mes", "temp_aire_C"]:
    y = df[v].interpolate()
    tren, prueba = y[:-24], y[-24:]
    m = SARIMAX(tren, order=(1,0,1), seasonal_order=(1,1,1,12)).fit(disp=False)
    pred = m.get_forecast(24).predicted_mean
    mae = (pred - prueba).abs().mean()
    final = SARIMAX(y, order=(1,0,1), seasonal_order=(1,1,1,12)).fit(disp=False)
    fc = final.get_forecast(12); ci = fc.conf_int()
    plt.figure(figsize=(14,4)); plt.plot(y[-72:], label="observado")
    plt.plot(pred, label=f"validación (MAE={mae:.2f})")
    plt.plot(fc.predicted_mean, "r", label="pronóstico 12 m"); plt.fill_between(ci.index, ci.iloc[:,0], ci.iloc[:,1], color="r", alpha=.15)
    plt.title(f"SARIMA · {v}"); plt.legend(); plt.show()

## 10. Predicción de humedad del suelo del próximo mes (Random Forest)
Útil para planear riego y fechas de siembra. Usa rezagos de 1–3 meses de todas las variables.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
objetivo = "hum_suelo_10_40cm_m3m3"
base = df.drop(columns=[c for c in df if c.startswith("anom_")]).interpolate()
X = pd.concat({f"{c}_lag{l}": base[c].shift(l) for c in base for l in (1,2,3)}, axis=1)
X["mes_sin"] = np.sin(2*np.pi*X.index.month/12); X["mes_cos"] = np.cos(2*np.pi*X.index.month/12)
datos = X.join(base[objetivo]).dropna()
corte = datos.index[-36]
Xtr, Xte = datos.loc[:corte].drop(columns=objetivo), datos.loc[corte:].drop(columns=objetivo)
ytr, yte = datos.loc[:corte, objetivo], datos.loc[corte:, objetivo]
rf = RandomForestRegressor(500, random_state=0).fit(Xtr, ytr)
p = rf.predict(Xte)
print(f"MAE={mean_absolute_error(yte,p):.4f}  R²={r2_score(yte,p):.3f}")
plt.figure(figsize=(14,4)); plt.plot(yte.index, yte, label="real"); plt.plot(yte.index, p, label="predicho"); plt.legend(); plt.title("Humedad de suelo 10–40 cm (últimos 36 meses)"); plt.show()
pd.Series(rf.feature_importances_, Xtr.columns).nlargest(15).plot(kind="barh", figsize=(8,6)); plt.title("Variables más importantes"); plt.show()

## 11. (Opcional) Descargar más datos directamente de NASA con `earthaccess`
Requiere tu usuario de **Earthdata Login** (el mismo de Giovanni). Te pedirá usuario y contraseña de forma interactiva; no los escribas en el código.

In [ ]:
!pip -q install earthaccess xarray netCDF4
import earthaccess, xarray as xr
earthaccess.login(strategy="interactive")
res = earthaccess.search_data(short_name="FLDAS_NOAH01_C_GL_M", temporal=("2025-01","2025-12"), bounding_box=(-99.5,18.33,-98.63,19.14))
archivos = earthaccess.open(res)
ds = xr.open_mfdataset(archivos).sel(X=slice(-99.5,-98.63), Y=slice(18.33,19.14))
ds["Rainf_f_tavg"].mean("time").plot(figsize=(7,5), cmap="Blues"); plt.title("Lluvia media 2025 sobre Morelos (kg m-2 s-1)"); plt.show()